# P3: Google Stock Price Prediction using LSTM (RNN)

In [ ]:
# Step 1: Install Libraries
!pip install numpy pandas matplotlib tensorflow scikit-learn

In [ ]:
# Step 2: Import Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

In [ ]:
# Step 3: Load Dataset
# FIX: Original code used dataset[['open']] but the CSV column is likely 'Open' (capital O).
# Always inspect column names before selecting.
dataset = pd.read_csv("Google_Stock_Price.csv")
print(dataset.columns.tolist())   # Inspect column names
print(dataset.tail(7))

In [ ]:
# Step 4: Select Feature (Open Price)
# FIX: Use the exact column name shown above. Most CSVs use 'Open' (capital O).
# FIX: Also remove commas from numbers if the CSV stores prices like '1,234.56'
dataset['Open'] = dataset['Open'].astype(str).str.replace(',', '').astype(float)

training_set = dataset[['Open']].values   # Shape: (n, 1)

In [ ]:
# Step 5: Feature Scaling (MinMax Normalization to [0, 1])
scaler = MinMaxScaler(feature_range=(0, 1))
training_set_scaled = scaler.fit_transform(training_set)

In [ ]:
# Step 6: Create Time Series Data (60-day lookback window)
# For each day i, the model sees the previous 60 days and predicts day i
X_train = []
y_train = []

for i in range(60, len(training_set_scaled)):
    X_train.append(training_set_scaled[i-60:i, 0])   # 60 previous prices
    y_train.append(training_set_scaled[i, 0])         # Price on day i

X_train = np.array(X_train)
y_train = np.array(y_train)

print("X_train shape before reshape:", X_train.shape)

In [ ]:
# Step 7: Reshape for LSTM
# LSTM expects input shape: (samples, timesteps, features)
X_train = np.reshape(X_train, (X_train.shape[0], X_train.shape[1], 1))
print("X_train shape after reshape:", X_train.shape)

In [ ]:
# Step 8: Build LSTM Model
model = Sequential()

model.add(Input(shape=(X_train.shape[1], 1)))          # (60 timesteps, 1 feature)

model.add(LSTM(units=50, return_sequences=True))        # Returns sequence for next LSTM
model.add(Dropout(0.2))                                 # Drop 20% neurons to reduce overfitting

model.add(LSTM(units=50, return_sequences=True))
model.add(Dropout(0.2))

model.add(LSTM(units=50, return_sequences=False))       # Last LSTM: no sequence output
model.add(Dropout(0.2))

model.add(Dense(units=1))                               # Output: single price value

model.summary()

In [ ]:
# Step 9: Compile Model
model.compile(optimizer='adam', loss='mean_squared_error')

In [ ]:
# Step 10: Train Model
history = model.fit(X_train, y_train, epochs=10, batch_size=32)

In [ ]:
# Step 11: Prepare Data for Prediction
# FIX: Original code used scaler.transform on the entire dataset.
# This is correct ONLY if the scaler was fit on all data — here we fit only on training,
# so we transform all data for generating the test windows.
dataset['Open'] = dataset['Open'].astype(str).str.replace(',', '').astype(float)  # Ensure clean
dataset_total = dataset[['Open']].values

inputs = scaler.transform(dataset_total)   # Scale using the already-fitted scaler

In [ ]:
# Step 12: Create Test Input
X_test = []

for i in range(60, len(inputs)):
    X_test.append(inputs[i-60:i, 0])

X_test = np.array(X_test)

# Reshape for LSTM: (samples, timesteps, features)
X_test = np.reshape(X_test, (X_test.shape[0], X_test.shape[1], 1))
print("X_test shape:", X_test.shape)

In [ ]:
# Step 13: Predict and Inverse Transform
predicted_stock_price = model.predict(X_test)
predicted_stock_price = scaler.inverse_transform(predicted_stock_price)  # Back to original scale

In [ ]:
# Step 14: Plot Real vs Predicted Stock Price
real_prices = dataset['Open'].values[60:]   # Skip first 60 (no prediction for those)

plt.figure(figsize=(12, 5))
plt.plot(real_prices, color='red',  label='Real Stock Price')
plt.plot(predicted_stock_price, color='blue', label='Predicted Stock Price')
plt.title('Google Stock Price Prediction (LSTM)')
plt.xlabel('Time (Days)')
plt.ylabel('Stock Price (USD)')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Step 15: Plot Training Loss
plt.figure(figsize=(8, 4))
plt.plot(history.history['loss'], label='Train Loss')
plt.title('Model Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.legend()
plt.show()